# 💡 Hint: Association Rules Metrics

In Market Basket Analysis, we represent a relationship as an association rule: **$X \rightarrow Y$**

---

### 1. Support
Support measures how frequently an itemset appears in the entire dataset.

$$ Support (X) = P(X) = \frac{\text{Number of transactions containing } X}{\text{Total number of transactions } (N)}$$

For two transactions, it is the probability that a transaction contains both $X$ and $Y$.

**Formula:**
$$Support(X \rightarrow Y) = P(X \cap Y) = \frac{\text{Number of transactions containing both } X \text{ and } Y}{\text{Total number of transactions } (N)}$$



---

### 2. Confidence
Confidence measures the reliability of the rule. It answers: "Of all people who bought $X$, what percentage also bought $Y$?"

**Formula:**
$$Confidence(X \rightarrow Y) = P(Y | X) = \frac{Support(X \cap Y)}{Support(X)}$$



---

### 3. Lift
Lift measures the strength of the association or the degree to which two items are dependent on each other, compared to what would be expected if they were independent.

**Formula:**
$$Lift(X \rightarrow Y) = \frac{Confidence(X \rightarrow Y)}{Support(Y)} = \frac{P(X \cap Y)}{P(X) \times P(Y)}$$

**Interpretation:**
* **Lift > 1:** Positive correlation ($X$ leads to $Y$).
* **Lift = 1:** Independent (No relationship).
* **Lift < 1:** Negative correlation (Buying $X$ makes $Y$ less likely).

**Part 1- A tiny dataset**-
Complete the code

In [1]:
import pandas as pd
import numpy as np

ratings_data = {
    'UserId': [1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 4],
    'MovieId': [1, 2, 3, 4, 5, 1, 3, 5, 2, 4, 5, 1, 2, 3, 5],
    'Rating': [5, 3, 4, 2, 5, 4, 5, 3, 2, 4, 5, 3, 4, 2, 5]
}

df = pd.DataFrame(ratings_data)

Watchlist = df.groupby('UserId')['MovieId'].apply(set).to_dict()
print("Transactions:", Watchlist)

Transactions: {1: {1, 2, 3, 4, 5}, 2: {1, 3, 5}, 3: {2, 4, 5}, 4: {1, 2, 3, 5}}


In [2]:
def calculate_support(item_set, Watchlist):
    count = 0
    for user_id, movies in Watchlist.items():
        if item_set.issubset(movies):
            count += 1
    return count / len(Watchlist)

sup_1_3 = calculate_support({1, 3}, Watchlist)
print(f"Support for {1, 3}: {sup_1_3}")

Support for (1, 3): 0.75


In [3]:
def rule_metrics(antecedent, consequent, transactions):
  # support, confidence and lift of the rule antecedent -> consequent (both are sets)
  support_antecedent = calculate_support(antecedent, transactions)
  support_consequent = calculate_support(consequent, transactions)
  support_both = calculate_support(antecedent | consequent, transactions)
  confidence = support_both / support_antecedent if support_antecedent > 0 else 0.0
  lift = confidence / support_consequent if support_consequent > 0 else 0.0
  return {"support": support_both, "confidence": confidence, "lift": lift}


def get_association_metrics(movie_a, movie_b, transactions):

  #### YOUR COD ###
  return rule_metrics({movie_a}, {movie_b}, transactions)


print("Rule 1 -> 3:", get_association_metrics(1, 3, Watchlist))
print("Rule 1 -> 4:", get_association_metrics(1, 4, Watchlist))

Rule 1 -> 3: {'support': 0.75, 'confidence': 1.0, 'lift': 1.3333333333333333}
Rule 1 -> 4: {'support': 0.25, 'confidence': 0.3333333333333333, 'lift': 0.6666666666666666}


In [4]:
def recommend_next_movie(current_basket, all_movie_ids, transactions, min_lift=1.0):

  ### YOUR CODE ###
  basket = set(current_basket)
  recommendations = []
  for movie in all_movie_ids:
    if movie in basket:
      continue
    metrics = rule_metrics(basket, {movie}, transactions)
    if metrics["lift"] > min_lift:
      recommendations.append((movie, metrics["confidence"], metrics["lift"]))
  return sorted(recommendations, key=lambda recommendation: recommendation[2], reverse=True)


all_movie_ids = sorted(df["MovieId"].unique().tolist())
print("Basket {1}:", recommend_next_movie({1}, all_movie_ids, Watchlist))
print("Basket {2}:", recommend_next_movie({2}, all_movie_ids, Watchlist))
print("Basket {1, 2}:", recommend_next_movie({1, 2}, all_movie_ids, Watchlist))

Basket {1}: [(3, 1.0, 1.3333333333333333)]
Basket {2}: [(4, 0.6666666666666666, 1.3333333333333333)]
Basket {1, 2}: [(3, 1.0, 1.3333333333333333)]


**Part 2**- This part of the lab uses a grocery-basket dataset. Each basket represents a transaction containing a set of purchased products.

In [5]:
import csv
import itertools
import math
import random
import time
from collections import Counter
from math import comb

import numpy as np
import pandas as pd
from IPython.display import display

DATA_PATH = "basket.csv"
SEED = 42
pd.set_option("display.width", 120)

import kagglehub
from pathlib import Path

DATA_PATH = str(
    Path(
        kagglehub.dataset_download(
            "rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba"
        )
    ) / "basket.csv"
)

In [6]:
from more_itertools import powerset
EPSILON = 0.001
K = 4

Task 1 — Check the first ten rows. See how products are represented and how individual transactions are separated.

**Question 1.** What does one row or basket represent in this dataset?

In [7]:
# Your Code
raw_baskets = pd.read_csv(DATA_PATH)
print("Shape:", raw_baskets.shape)
raw_baskets.head(10)

Shape: (14963, 11)


,0,1,2,3,4,5,6,7,8,9,10
0,whole milk,pastry,salty snack,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sausage,whole milk,semi-finished bread,yogurt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,soda,pickled vegetables,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,canned beer,misc. beverages,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,sausage,hygiene articles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,sausage,whole milk,rolls/buns,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,whole milk,soda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,frankfurter,soda,whipped/sour cream,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,frankfurter,curd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,beef,white bread,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Your answer (Q1):

One row is one basket: all products that one customer bought on one day. There are 14963 baskets. Each product is stored as its name in a separate column (`0` to `10`), so a row can hold at most 11 products. Rows have different lengths (from 2 to 11 entries), so the rest of the row is filled with empty cells (NaN). The same product can appear twice in one row, so as a set of distinct products a basket has 1 to 10 items. The header row holds only the column numbers. Transactions are separated simply by being on different rows.

## Task 2 — Read transactions and identify products

Implement the following functions:

- `read_baskets(path)`: read the file and return a list of baskets (sets of products).

- The file has a header row. Skip it.
- The first column is a basket id, not a product. Ignore it.
- Convert product names to lowercase and strip spaces.
- Omit empty (NaN) entries.
- Keep only non-empty baskets, in file order.
- `unique_products(baskets)`: return a sorted list of all distinct products that occur in the baskets.

Then run the code to inspect the baskets and product list.

In [8]:
def read_baskets(path: str) -> list[set[str]]:
    # TODO: Read the file and convert each transaction into a set of lowercase product names.
    # basket.csv has no basket id column: column 0 already holds the first product, so no column is skipped.
    with open(path, newline="", encoding="utf-8") as file:
        rows = csv.reader(file)
        next(rows)
        baskets = [{product.strip().lower() for product in row if product.strip()} for row in rows]
    return [basket for basket in baskets if basket]

def unique_products(baskets: list[set[str]]) -> list[str]:
    # TODO: Collect all distinct products and return them in sorted order.
    return sorted(set().union(*baskets))

baskets = read_baskets(DATA_PATH)
products = unique_products(baskets)

print("Number of baskets:", len(baskets))
print("First few baskets:", baskets[:5])
print("Number of distinct products:", len(products))
print("First few products:", products[:20])

Number of baskets: 14963
First few baskets: [{'salty snack', 'whole milk', 'pastry'}, {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}, {'soda', 'pickled vegetables'}, {'misc. beverages', 'canned beer'}, {'hygiene articles', 'sausage'}]
Number of distinct products: 167
First few products: ['abrasive cleaner', 'artif. sweetener', 'baby cosmetics', 'bags', 'baking powder', 'bathroom cleaner', 'beef', 'berries', 'beverages', 'bottled beer', 'bottled water', 'brandy', 'brown bread', 'butter', 'butter milk', 'cake bar', 'candles', 'candy', 'canned beer', 'canned fish']


## Task 3 — Encode products and calculate support

Implement the functions below.

1. `get_products_id`: assign each distinct product a unique integer ID.
2. `encode_basket`: convert a basket into a tuple of product IDs in a consistent order.
3. `get_supports`: calculate the support of itemsets found in the transactions. Use `EPSILON` as the minimum support threshold and consider itemsets up to size `K`.

After implementing the functions, inspect the resulting support dictionary.

In [9]:
def get_products_id(all_products: list[str]) -> dict[str, int]:
    # TODO
    return {product: product_id for product_id, product in enumerate(all_products)}


def encode_basket(basket: set[str], products_id: dict[str, int]) -> tuple[int, ...]:
    # TODO
    return tuple(sorted(products_id[product] for product in basket))


def get_supports(baskets, all_products, products_id, epsilon):

    # TODO
    counts = Counter()
    for basket in baskets:
        for itemset in powerset(encode_basket(basket, products_id)):
            if 1 <= len(itemset) <= K:
                counts[itemset] += 1

    n_baskets = len(baskets)
    return {
        itemset: count / n_baskets
        for itemset, count in counts.items()
        if count / n_baskets >= epsilon
    }



PRODUCTS_ID = get_products_id(products)
supports = get_supports(baskets, products, PRODUCTS_ID, EPSILON)
print("Number of supported itemsets:", len(supports))
list(supports.items())[:10]

Number of supported itemsets: 750


[((104,), 0.0517275947336764),
 ((127,), 0.018779656485998796),
 ((164,), 0.15792287642852368),
 ((104, 164), 0.006482657221145492),
 ((127, 164), 0.0019381140145692708),
 ((129,), 0.06034886052262247),
 ((131,), 0.009490075519615051),
 ((165,), 0.08587850030074183),
 ((129, 164), 0.008955423377664907),
 ((129, 165), 0.005747510525964045)]

## Task 4 — Calculate support, confidence, and lift

In [10]:
def support(supports, products: set[str], products_id: dict[str, int]) -> float:
    # TODO: Return the stored support for this itemset.
    # Decide how your implementation should handle an itemset that is not in supports.
    # An itemset missing from supports is rarer than EPSILON, so we treat its support as 0.
    return supports.get(encode_basket(products, products_id), 0.0)


def confidence(
    supports,
    prior_products: set[str],
    following_products: set[str],
    products_id: dict[str, int]
) -> float:
    # TODO: Calculate confidence for prior_products -> following_products.
    prior_support = support(supports, prior_products, products_id)
    if prior_support == 0:
        return 0.0
    return support(supports, prior_products | following_products, products_id) / prior_support


def lift(
    supports,
    prior_products: set[str],
    following_products: set[str],
    products_id: dict[str, int]
) -> float:
    # TODO: Calculate lift for prior_products -> following_products.
    following_support = support(supports, following_products, products_id)
    if following_support == 0:
        return 0.0
    return confidence(supports, prior_products, following_products, products_id) / following_support

In [11]:
# Test your implementations on this example.
antecedent = {"whole milk", "rolls/buns"}
consequent = {"yogurt"}

print("Support:", support(supports, antecedent | consequent, PRODUCTS_ID))
print("Confidence:", confidence(supports, antecedent, consequent, PRODUCTS_ID))
print("Lift:", lift(supports, antecedent, consequent, PRODUCTS_ID))

Support: 0.0013366303548753592
Confidence: 0.09569377990430622
Lift: 1.1142926293448514


## Task 5 — Generate recommendation candidates

Implement two candidate-generation strategies.

Both functions should:
1. consider products not already present in the current basket;
2. calculate the confidence and lift for adding each candidate product;
3. keep candidates with lift greater than 1;
4. return at most ten candidates.

Return each candidate as a tuple:

`(recommended_product, current_basket, confidence, lift)`

**Basic strategy:** rank candidates by confidence, in descending order.

**Advanced strategy:** rank candidates by the product of confidence and lift, in descending order.

After implementation, compare the results and consider how the ranking criterion changes the recommendations.

In [12]:
def rule_candidates(
    basket: set[str],
    products: list[str],
    supports,
    products_id: dict[str, int]
) -> list[tuple[str, set[str], float, float]]:
    # Every rule prior -> {product} with lift > 1, where prior is a subset of the basket.
    # The whole basket is usually too rare to be in supports, so we use its subsets of up to K - 1 products.
    candidates = []
    for prior in powerset(sorted(basket)):
        if not 1 <= len(prior) < K:
            continue
        prior = set(prior)
        for product in products:
            if product in basket:
                continue
            rule_lift = lift(supports, prior, {product}, products_id)
            if rule_lift > 1:
                candidates.append((product, basket, confidence(supports, prior, {product}, products_id), rule_lift))
    return candidates

def top_candidates(candidates, score, n=10):
    # Keep the best rule for each product, highest score first.
    best = {}
    for candidate in sorted(candidates, key=score, reverse=True):
        best.setdefault(candidate[0], candidate)
    return list(best.values())[:n]

def generate_basic_candidates(
    basket: set[str],
    products: list[str],
    supports,
    products_id: dict[str, int]
) -> list[tuple[str, set[str], float, float]]:
    # TODO: Generate eligible candidates and rank by confidence (descending).
    candidates = rule_candidates(basket, products, supports, products_id)
    return top_candidates(candidates, score=lambda candidate: candidate[2])

def generate_advanced_candidates(
    basket: set[str],
    products: list[str],
    supports,
    products_id: dict[str, int]
) -> list[tuple[str, set[str], float, float]]:
    # TODO: Generate eligible candidates and rank by confidence * lift (descending).
    candidates = rule_candidates(basket, products, supports, products_id)
    return top_candidates(candidates, score=lambda candidate: candidate[2] * candidate[3])

## Task 6 — Test and interpret recommendations

Run the candidate-generation functions on the two example baskets below.

Answer these questions:

2. Which products are recommended for each basket?
3. Are the basic and advanced rankings different? If so, why?
4. Why might a high-confidence rule not necessarily have a high lift?
5. What are two limitations of using association rules for recommendations?

In [13]:
print("Basket:", baskets[1])
print("\nBasic recommendations:")
print(generate_basic_candidates(baskets[1], products, supports, PRODUCTS_ID))
print("\nAdvanced recommendations:")
print(generate_advanced_candidates(baskets[1], products, supports, PRODUCTS_ID))

Basket: {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}

Basic recommendations:
[('rolls/buns', {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}, 0.12686567164179105, 1.1532752398396837), ('soda', {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}, 0.11940298507462688, 1.2296124333596987), ('bottled beer', {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}, 0.05537098560354374, 1.2220000849348451), ('citrus fruit', {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}, 0.053696498054474705, 1.0106423904265471), ('pastry', {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}, 0.05315614617940199, 1.0276168156103256), ('curd', {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}, 0.04872646733111849, 1.4466153386419167), ('frozen vegetables', {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}, 0.03433001107419712, 1.2259664813919129), ('beverages', {'semi-finished bread', 'whole milk', 'sausage', 'yogurt'}, 0.02547065337763012

In [14]:
print("Basket:", baskets[33])
print("\nBasic recommendations:")
print(generate_basic_candidates(baskets[33], products, supports, PRODUCTS_ID))
print("\nAdvanced recommendations:")
print(generate_advanced_candidates(baskets[33], products, supports, PRODUCTS_ID))

Basket: {'root vegetables', 'photo/film', 'soda', 'tropical fruit', 'yogurt', 'white wine', 'domestic eggs'}

Basic recommendations:
[('sausage', {'root vegetables', 'photo/film', 'soda', 'tropical fruit', 'yogurt', 'white wine', 'domestic eggs'}, 0.0669260700389105, 1.1089864739670185), ('citrus fruit', {'root vegetables', 'photo/film', 'soda', 'tropical fruit', 'yogurt', 'white wine', 'domestic eggs'}, 0.053696498054474705, 1.0106423904265471), ('shopping bags', {'root vegetables', 'photo/film', 'soda', 'tropical fruit', 'yogurt', 'white wine', 'domestic eggs'}, 0.04803073967339097, 1.0093875810856028), ('newspapers', {'root vegetables', 'photo/film', 'soda', 'tropical fruit', 'yogurt', 'white wine', 'domestic eggs'}, 0.04144144144144145, 1.0654437943097739), ('coffee', {'root vegetables', 'photo/film', 'soda', 'tropical fruit', 'yogurt', 'white wine', 'domestic eggs'}, 0.03783783783783784, 1.1969716016227645), ('frankfurter', {'root vegetables', 'photo/film', 'soda', 'tropical fruit

**Note on Task 5.** A rule built from the whole basket never has enough support. `supports` holds no itemset with more than 3 products, and both example baskets are longer, so such rules would give an empty list. The confidence and lift of each candidate therefore come from its best rule `prior -> product`, where `prior` is a subset of the basket with up to `K - 1` products. The tuple still holds the current basket, as the task asks.

**Your answer (Q2):**

Both example baskets are too rare as a whole to be in `supports`, so the rules use subsets of the basket (up to `K - 1` products) as the prior. Each product is ranked by its best rule.

Basket 1 (`sausage`, `semi-finished bread`, `whole milk`, `yogurt`):
* basic: rolls/buns, soda, bottled beer, citrus fruit, pastry, curd, frozen vegetables, beverages, dessert, frozen meals;
* advanced: soda, rolls/buns, curd, bottled beer, pastry, citrus fruit, frozen vegetables, beverages, frozen meals, sliced cheese.

The two strongest rules come from `{sausage, whole milk}`: rolls/buns (confidence 0.127, lift 1.15) and soda (confidence 0.119, lift 1.23). Most other rules come from `sausage` alone.

Basket 33 (`domestic eggs`, `photo/film`, `root vegetables`, `soda`, `tropical fruit`, `white wine`, `yogurt`):
* basic: sausage, citrus fruit, shopping bags, newspapers, coffee, frankfurter, frozen vegetables, white bread, uht-milk, specialty chocolate;
* advanced: sausage, citrus fruit, shopping bags, coffee, newspapers, frankfurter, frozen vegetables, white bread, flour, specialty chocolate.

Here every rule has a single product as its prior. The best one is yogurt → sausage (confidence 0.067, lift 1.11).

**Your answer (Q3):**

The rankings are different, but only a little. For each basket, 9 of the 10 products are the same, and the changes are swaps between neighbours plus one product at the bottom. In basket 1, curd moves up from 6th to 3rd place, and dessert (lift 1.03) is replaced by sliced cheese (lift 1.34). In basket 33, uht-milk (lift 1.06) is replaced by flour (lift 1.62).

The basic strategy ranks by confidence only, and confidence is high for products that are popular anyway. The advanced strategy multiplies confidence by lift, so it rewards products whose link with the basket is stronger than their general popularity. The difference is small here because all confidences are low (0.02–0.13) and all lifts are close to 1 (1.0–1.6). Multiplying by such a lift changes the score only a little, so the products mostly keep their places.

**Your answer (Q4):**

Confidence is P(Y | X). It is high whenever Y is popular, even if X has nothing to do with Y. Lift divides confidence by P(Y), so it measures how much X raises the chance of Y compared with a random customer. For example, rolls/buns are bought in 11% of all baskets. The rule {sausage, whole milk} → rolls/buns has the highest confidence in basket 1 (0.127), but its lift is only 1.15, because buying sausage and milk makes rolls/buns only slightly more likely than usual. In the extreme case, a product bought by every customer would have confidence 1.0 for every rule but lift exactly 1, which means no association at all.

**Your answer (Q5):**

1. Rare products and long baskets are lost. An itemset below the support threshold has no rules at all. 18 of the 167 products are too rare to pass the threshold, so they can never be recommended (this also applies to every new product). Full baskets with 4 or more products never pass it either, so the rules can only use small parts of the basket and ignore the rest.
2. The rules are generic, not personal. They only describe which products are bought together on average, and they ignore who the customer is, their past purchases, quantities, prices and the time of purchase. Two customers with the same basket always get the same list. The rules also favour popular products, and in this dataset they are weak: the best confidence is only 0.127.